# Scam Message Detector (simple version, trained on old + new data)

Steps: 1) load the old UCI data, 2) upload your newer datasets, 3) merge and clean, 4) train one model on everything, 5) check that new data actually helped, 6) try it.

Run it top to bottom (`Runtime > Run all`). Step 2 opens a file picker although they are uploaded already on github so the link uses them. If something were to happen, the upload button still pops up.

In [1]:
!pip install -q gradio

import io, os, re, zipfile, unicodedata, urllib.request
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SEED = 42

## Text clean-up (anti-evasion)
Scammers swap letters for lookalikes (Cyrillic `а` instead of Latin `a`) or hide zero-width characters inside words. Before the model sees any message, it is cleaned: Unicode NFKC normalization, zero-width characters removed, common Cyrillic lookalikes mapped back to Latin. This runs automatically inside the model (training, testing and single messages).

In [2]:
CONFUSABLES = str.maketrans({
    "\u0430": "a", "\u0435": "e", "\u043e": "o", "\u0440": "p", "\u0441": "c", "\u0445": "x",
    "\u0410": "A", "\u0415": "E", "\u041e": "O", "\u0420": "P", "\u0421": "C", "\u0425": "X",
})
ZERO_WIDTH = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2060\ufeff]")

def normalize_for_inference(text):
    text = unicodedata.normalize("NFKC", str(text)).translate(CONFUSABLES)
    return ZERO_WIDTH.sub("", text)

def normalize_many(texts):
    return [normalize_for_inference(t) for t in texts]

# quick demo: the first message uses Cyrillic lookalike letters
before = "Y\u043eur \u0430cc\u043eunt is l\u043eck\u0435d. V\u0435rify now"
print(before, "->", normalize_for_inference(before))

Yоur аccоunt is lоckеd. Vеrify now -> Your account is locked. Verify now


## 1. Old data (UCI SMS Spam Collection)

In [3]:
URL = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
raw = urllib.request.urlopen(URL, timeout=30).read()
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    with z.open('SMSSpamCollection') as f:
        uci = pd.read_csv(f, sep='\t', header=None, names=['label', 'text'])

uci['is_scam'] = (uci['label'] == 'spam').astype(int)
uci = uci[['text', 'is_scam']].drop_duplicates(subset='text').reset_index(drop=True)
uci['source'] = 'UCI-2011'
print(len(uci), 'old messages,', f"{uci['is_scam'].mean():.1%} scams")

5169 old messages, 12.6% scams


## 2. Get the new data
If `DATA_BASE_URL` is set (the files are hosted online), they download automatically and reviewers do nothing. If it is empty, a file picker opens and you choose the CSV files from your computer.

In [4]:
DATA_BASE_URL = "https://raw.githubusercontent.com/TaranAzrot-13/scam-detector/main/data/"
DATA_FILES = ['Dataset_5971.csv', 'smishtank.csv', 'modern_test_messages.csv', 'locked_test_messages.csv']

if DATA_BASE_URL:
    for f in DATA_FILES:
        if not os.path.exists(f):
            try:
                urllib.request.urlretrieve(DATA_BASE_URL + f, f)
            except Exception as e:
                print('Could not download', f, '->', e)
    print('Data files ready:', [f for f in DATA_FILES if os.path.exists(f)])
else:
    print('DATA_BASE_URL is empty, so choose the files from your computer instead:')
    from google.colab import files
    files.upload()

Data files ready: ['Dataset_5971.csv', 'smishtank.csv', 'modern_test_messages.csv', 'locked_test_messages.csv']


In [5]:
# These are the file names you can copy into the list below
print([f for f in os.listdir('.') if f.lower().endswith(('.csv', '.tsv', '.txt'))])

['locked_test_messages.csv', 'modern_test_messages.csv', 'smishtank.csv', 'Dataset_5971.csv']


## 3. Tell the notebook which files to use
Edit this list: `(file name, a short name for the source, default label)`.
- Default label `None` means the file has a label column (spam/ham, 0/1, ...) and it is detected automatically.
- Default label `1` means **every** message in the file is a scam (a scam-only dataset like SmishTank).

**Important:** if your new data is scam-only, also add some *normal* modern messages (e.g. `modern_test_messages.csv` has both). Otherwise the model may learn that anything that sounds modern is a scam.

In [13]:
NEW_FILES = [
    ('modern_test_messages.csv', 'synthetic-modern', None),
    ('SMSPhishingDataset.csv', 'mendeley', None),
    ('smishtank.csv', 'smishtank', 1),
]
MAX_PER_SOURCE = 3000   # keeps training fast and sources balanced

TEXT_NAMES = ['text', 'message', 'sms', 'content', 'body', 'msg', 'message_text', 'smstext', 'maintext', 'main_text', 'sms_text']
LABEL_NAMES = ['label', 'class', 'type', 'category', 'target', 'spam', 'is_spam', 'v1']
SCAM = {'spam', 'smish', 'smishing', 'phishing', 'scam', 'fraud', 'malicious', '1', '1.0', 'true', 'yes'}
NORMAL = {'ham', 'legitimate', 'legit', 'normal', 'benign', 'safe', '0', '0.0', 'false', 'no'}

def load_new(path, source, default_label=None):
    table = None
    for enc in ('utf-8', 'latin-1'):
        try:
            table = pd.read_csv(path, sep=None, engine='python', encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    cols = {str(c).lower().strip(): c for c in table.columns}
    tcol = next((cols[c] for c in TEXT_NAMES if c in cols), None)
    if tcol is None:
        raise ValueError(f'{path}: no text column found. Columns are {list(table.columns)}. Rename the text column to text.')
    table = table.dropna(subset=[tcol])

    if default_label is not None:
        label = pd.Series(default_label, index=table.index)
    else:
        lcol = next((cols[c] for c in LABEL_NAMES if c in cols), None)
        if lcol is None:
            raise ValueError(f'{path}: no label column found. Columns are {list(table.columns)}. Rename it to label, or use default label 1 for scam-only files.')
        norm = table[lcol].astype(str).str.strip().str.lower()
        label = norm.map(lambda v: 1 if v in SCAM else (0 if v in NORMAL else np.nan))

    out = pd.DataFrame({'text': table[tcol].astype(str).str.strip(), 'is_scam': label}).dropna()
    out['is_scam'] = out['is_scam'].astype(int)
    out = out[out['text'].str.len() > 0].drop_duplicates(subset='text')
    out['source'] = source
    return out.reset_index(drop=True)

loaded = []
for fname, source, default_label in NEW_FILES:
    if Path(fname).exists():
        loaded.append(load_new(fname, source, default_label))
    else:
        print('Skipping', fname, '(not uploaded)')

if loaded:
    new = pd.concat(loaded, ignore_index=True)
    new = new[~new['text'].isin(set(uci['text']))].drop_duplicates(subset='text')
    new = pd.concat([g.sample(min(len(g), MAX_PER_SOURCE), random_state=SEED) for _, g in new.groupby('source')], ignore_index=True)
    data = pd.concat([uci, new], ignore_index=True)
else:
    print('No new data loaded, so this will only use the old data.')
    data = uci.copy()

summary = data.groupby('source')['is_scam'].agg(messages='size', scams='sum')
summary['normal'] = summary['messages'] - summary['scams']
summary

Skipping SMSPhishingDataset.csv (not uploaded)


,messages,scams,normal
source,,,
UCI-2011,5169,653,4516
smishtank,1055,1055,0
synthetic-modern,100,50,50


## 4. Train
Two models on the same split: **old-only** (trained on the UCI part of the training set) and **old + new** (trained on everything in the training set). The model reads both whole words and character patterns, so it copes better with odd spellings and new wording.

In [7]:
train, test = train_test_split(data, test_size=0.2, random_state=SEED, stratify=data['is_scam'])

def make_model():
    features = FeatureUnion([
        ('words', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ('chars', TfidfVectorizer(lowercase=True, analyzer='char_wb', ngram_range=(2, 5), min_df=3,
                                  sublinear_tf=True, max_features=200000)),
    ])
    return Pipeline([('normalize', FunctionTransformer(normalize_many)),
                     ('features', features),
                     ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))])

old_train = train[train['source'] == 'UCI-2011']
old_model = make_model().fit(old_train['text'], old_train['is_scam'])
new_model = make_model().fit(train['text'], train['is_scam'])
print('Both models trained on', len(old_train), '(old only) and', len(train), '(old + new) messages.')

Both models trained on 4132 (old only) and 4215 (old + new) messages.


## 5. Did the new data help?
Per source, on held-out test messages neither model saw. **Scams caught** should be high, **false alarms** (normal messages wrongly flagged) should be low. Look at the new sources: the old-only model should do worse there.

In [8]:
rows = []
for src, g in test.groupby('source'):
    y = g['is_scam'].values
    row = {'source': src, 'n': len(g)}
    for name, m in [('old only', old_model), ('old + new', new_model)]:
        pred = m.predict(g['text'])
        row[f'scams caught ({name})'] = round(float(pred[y == 1].mean()), 3) if (y == 1).any() else None
        row[f'false alarms ({name})'] = round(float(pred[y == 0].mean()), 3) if (y == 0).any() else None
    rows.append(row)
pd.DataFrame(rows).set_index('source')

,n,scams caught (old only),false alarms (old only),scams caught (old + new),false alarms (old + new)
source,,,,,
UCI-2011,1037,0.962,0.006,0.962,0.008
synthetic-modern,17,0.625,0.111,0.750,0.111


## 6. Try it
Edit the example messages or add your own. The words shown are the ones that pushed the score towards scam.

In [9]:
def check(text, model=new_model, top_n=5):
    text = normalize_for_inference(text)
    prob = model.predict_proba([text])[0, 1]
    feats = model.named_steps['features']
    coefs = model.named_steps['clf'].coef_[0]
    names = feats.get_feature_names_out()
    vec = feats.transform([text])
    contrib = {names[i][len('words__'):]: coefs[i] * vec[0, i]
               for i in vec.nonzero()[1] if names[i].startswith('words__')}
    triggers = [w for w, c in sorted(contrib.items(), key=lambda kv: -kv[1]) if c > 0][:top_n]
    return round(float(prob), 2), triggers

for t in [
    'Your parcel is on hold. Pay a small fee here to reschedule delivery: http://bit.ly/xyz',
    'Your bank account has been locked. Verify your identity within 24 hours to avoid closure',
    'Hi, your appointment is confirmed for Tuesday at 3pm. See you then!',
    'Can you pick up some milk on the way home?',
]:
    print(check(t), '<-', t[:70])

(0.8, ['http', 'your', 'to', 'delivery', 'pay']) <- Your parcel is on hold. Pay a small fee here to reschedule delivery: h
(0.73, ['your', 'account', 'to', 'been', 'has been']) <- Your bank account has been locked. Verify your identity within 24 hour
(0.1, ['your', 'for', 'is', 'hi']) <- Hi, your appointment is confirmed for Tuesday at 3pm. See you then!
(0.02, ['on', 'some', 'the']) <- Can you pick up some milk on the way home?


### Check your own single message
No CSV needed: type or paste one message and run this cell. It asks for the message in a box.

In [10]:
message = input("Paste one message: ")
prob, triggers = check(message)
verdict = "Likely a scam" if prob >= 0.75 else ("Suspicious" if prob >= 0.40 else "Probably fine")
print(f"{verdict}  (score {int(prob * 100)}%)")
print("Words that raised suspicion:", ", ".join(triggers) if triggers else "none")

Paste one message: pay for a delivery
Suspicious  (score 51%)
Words that raised suspicion: delivery, for, pay, for delivery


## 7. Optional: locked final test
Upload `locked_test_messages.csv` and run this cell **once, at the very end**, after you stop changing the model. It reports the number to quote in your README.

In [11]:
if Path('locked_test_messages.csv').exists():
    locked = pd.read_csv('locked_test_messages.csv')
    locked = locked[~locked['text'].isin(set(data['text']))].reset_index(drop=True)
    y = locked['label'].values
    out = []
    for name, m in [('old only', old_model), ('old + new', new_model)]:
        p = m.predict_proba(locked['text'])[:, 1]
        out.append({'model': name, 'AUC': round(roc_auc_score(y, p), 3),
                    'scams flagged (>=0.5)': round(float((p[y == 1] >= 0.5).mean()), 2),
                    'normal flagged (>=0.5)': round(float((p[y == 0] >= 0.5).mean()), 2)})
    print('Locked test, n =', len(locked))
    display(pd.DataFrame(out).set_index('model'))
else:
    print('locked_test_messages.csv not found. Upload it (rerun the upload cell) and run this once at the end.')

Locked test, n = 100


,AUC,scams flagged (>=0.5),normal flagged (>=0.5)
model,,,
old only,0.757,0.58,0.22
old + new,0.869,0.84,0.30


## 8. Optional: web demo
Creates a temporary public link. Do not paste private messages into it.

In [12]:
import gradio as gr

def analyse(text):
    if not text or not text.strip():
        return 'Paste a message above.'
    prob, triggers = check(text.strip()[:2000])
    if prob >= 0.75:
        verdict = 'Likely a scam'
    elif prob >= 0.40:
        verdict = 'Suspicious'
    else:
        verdict = 'Probably fine'
    why = ', '.join(triggers) if triggers else 'no strong warning words found'
    return f'**{verdict}** (score {int(prob * 100)}%)\n\nWords that raised suspicion: {why}\n\n_A helper, not a guarantee._'

gr.Interface(fn=analyse, inputs=gr.Textbox(lines=6, label='Paste the message'),
             outputs=gr.Markdown(), title='Scam Message Detector').launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bac747ddec26344943.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
